## 1. IMPORTACIONES

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

## 2. CONFIGURACIÓN DEL PROYECTO

In [2]:
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import (
    GOLD_DIMENSIONS_DIR,
    GOLD_METRICS_DIR,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GOLD_DIMENSIONS_DIR:", GOLD_DIMENSIONS_DIR)
print("GOLD_METRICS_DIR:", GOLD_METRICS_DIR)

PROJECT_ROOT: d:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch
GOLD_DIMENSIONS_DIR: D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\gold\analytics\dimensional\dimensions
GOLD_METRICS_DIR: D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\gold\analytics\metrics


## 3. CARGA DE GOLD ANALYTICS

In [5]:
dim_movie = pd.read_parquet(
    GOLD_DIMENSIONS_DIR / "dim_movie.parquet"
)

movie_metrics = pd.read_parquet(
    GOLD_METRICS_DIR / "movie_metrics.parquet"
)

print("dim_movie:", dim_movie.shape)
print("movie_metrics:", movie_metrics.shape)

dim_movie: (87585, 19)
movie_metrics: (84432, 4)


## 4. INTEGRACION DE CATALOGO + METRICAS

In [6]:
baseline_df = dim_movie.merge(
    movie_metrics,
    on="movie_id",
    how="left",
    validate="one_to_one",
)

baseline_df[
    [
        "movie_id",
        "title",
        "rating_count",
        "rating_mean",
        "rating_std",
    ]
].head()

,movie_id,title,rating_count,rating_mean,rating_std
0,1,Toy Story,68997.0,3.897438,0.921450
1,2,Jumanji,28904.0,3.275758,0.955544
2,3,Discordias a la carta,13134.0,3.139447,1.012273
3,4,Esperando un respiro,2806.0,2.845331,1.105871
4,5,Vuelve el padre de la novia (Ahora también abu...,13154.0,3.059602,0.999079


## 5. BASELINE POR POPULARIDAD

In [8]:
popular_top = (
    baseline_df
    .dropna(
        subset=[
            "rating_count",
            "title",
        ]
    )
    .sort_values(
        by="rating_count",
        ascending=False,
    )
    .head(10)
)

popular_top[
    [
        "title",
        "rating_count",
        "rating_mean",
    ]
]

,title,rating_count,rating_mean
314,"Shawshank Redemption, The (1994)",102929.0,4.404614
351,Forrest Gump (1994),100296.0,4.052744
292,Pulp Fiction (1994),98409.0,4.196969
2480,"Matrix, The (1999)",93808.0,4.156437
585,"Silence of the Lambs, The (1991)",90330.0,4.148367
257,Star Wars: Episode IV - A New Hope (1977),85010.0,4.099824
2867,Fight Club (1999),77332.0,4.228780
475,Jurassic Park (1993),75233.0,3.698623
522,Schindler's List (1993),73849.0,4.236990
4888,"Lord of the Rings: The Fellowship of the Ring,...",73122.0,4.092134


## 6. BASELINE POR RATING PROMEDIO

In [9]:
rating_top = (
    baseline_df
    .dropna(
        subset=[
            "rating_mean",
            "title",
        ]
    )
    .sort_values(
        by="rating_mean",
        ascending=False,
    )
    .head(10)
)

rating_top[
    [
        "title",
        "rating_mean",
        "rating_count",
    ]
]

,title,rating_mean,rating_count
84409,Shadows (2023),5.0,1.0
84407,Mal de Ojo (2022),5.0,1.0
84386,Mr. Krueger's Christmas (1980),5.0,1.0
84385,It's a Wonderful Life (2007),5.0,1.0
56921,Traces of Smoke (1992),5.0,2.0
56935,Love Pret-a-porte (2017),5.0,1.0
57179,The Revolution That Wasn't (2008),5.0,1.0
57017,Still Burning (2016),5.0,1.0
84361,ManFish (2022),5.0,1.0
84334,Going Varsity In Mariachi,5.0,1.0


#### Como los resultados no son viables por que tiene una valoración de 5 pero solo una valoracion, no es un rating muy fiable, es por esto que trabajaremos con un minimo de 100 valoraciones para tener un resultado mas real

## 7. APLICACION DE UN MINIMO DE RATING

In [11]:
movie_metrics["rating_count"].describe()
MIN_RATINGS = 100

In [14]:
rating_filtered_top = (
    baseline_df[
        baseline_df["rating_count"] >= MIN_RATINGS
    ]
    .sort_values(
        by="rating_mean",
        ascending=False,
    )
    .head(10)
)

rating_filtered_top[
    [
        "title",
        "rating_mean",
        "rating_count",
    ]
]

,title,rating_mean,rating_count
46252,Planet Earth II (2016),4.446830,1956.0
40985,Planet Earth (2006),4.444369,2948.0
46103,Band of Brothers (2001),4.426539,2811.0
314,"Shawshank Redemption, The (1994)",4.404614,102929.0
46487,Cosmos,4.330081,615.0
840,"Godfather, The (1972)",4.317030,66440.0
61090,Parasite (2019),4.312254,11670.0
50126,Blue Planet II (2017),4.300086,1163.0
59131,Twin Peaks (1989),4.298684,1140.0
68471,Twelve Angry Men (1954),4.286192,449.0


## 8. BASELINE PONDEERADO

In [18]:
C = baseline_df["rating_mean"].mean()

m = baseline_df["rating_count"].quantile(0.90)

C = rating medio global. 

m = mínimo de ratings para considerar popularidad relevante.

In [19]:
qualified = baseline_df[
    baseline_df["rating_count"] >= m
].copy()

In [20]:
qualified["weighted_score"] = (
    (
        qualified["rating_count"]
        / (
            qualified["rating_count"] + m
        )
    )
    * qualified["rating_mean"]
    +
    (
        m
        / (
            qualified["rating_count"] + m
        )
    )
    * C
)

In [22]:
weighted_top = (
    qualified
    .sort_values(
        by="weighted_score",
        ascending=False,
    )
    .head(10)
)

weighted_top[
    [
        "title",
        "rating_mean",
        "rating_count",
        "weighted_score",
    ]
]

,title,rating_mean,rating_count,weighted_score
314,"Shawshank Redemption, The (1994)",4.404614,102929.0,4.401238
40985,Planet Earth (2006),4.444369,2948.0,4.332311
840,"Godfather, The (1972)",4.317030,66440.0,4.312134
46103,Band of Brothers (2001),4.426539,2811.0,4.310914
61090,Parasite (2019),4.312254,11670.0,4.284956
46252,Planet Earth II (2016),4.446830,1956.0,4.284079
49,"Usual Suspects, The (1995)",4.265070,67750.0,4.260458
1190,"Godfather: Part II, The (1974)",4.264468,43111.0,4.257239
1173,12 Angry Men (1957),4.265311,21863.0,4.251126
522,Schindler's List (1993),4.236990,73849.0,4.232852


## 9. COMPARAR LOS TRES BASELINES

In [23]:
print("TOP POPULARIDAD")
display(
    popular_top[
        [
            "title",
            "rating_count",
            "rating_mean",
        ]
    ]
)

print("\nTOP RATING FILTRADO")
display(
    rating_filtered_top[
        [
            "title",
            "rating_mean",
            "rating_count",
        ]
    ]
)

print("\nTOP WEIGHTED SCORE")
display(
    weighted_top[
        [
            "title",
            "weighted_score",
            "rating_mean",
            "rating_count",
        ]
    ]
)

TOP POPULARIDAD


,title,rating_count,rating_mean
314,"Shawshank Redemption, The (1994)",102929.0,4.404614
351,Forrest Gump (1994),100296.0,4.052744
292,Pulp Fiction (1994),98409.0,4.196969
2480,"Matrix, The (1999)",93808.0,4.156437
585,"Silence of the Lambs, The (1991)",90330.0,4.148367
257,Star Wars: Episode IV - A New Hope (1977),85010.0,4.099824
2867,Fight Club (1999),77332.0,4.228780
475,Jurassic Park (1993),75233.0,3.698623
522,Schindler's List (1993),73849.0,4.236990
4888,"Lord of the Rings: The Fellowship of the Ring,...",73122.0,4.092134



TOP RATING FILTRADO


,title,rating_mean,rating_count
46252,Planet Earth II (2016),4.446830,1956.0
40985,Planet Earth (2006),4.444369,2948.0
46103,Band of Brothers (2001),4.426539,2811.0
314,"Shawshank Redemption, The (1994)",4.404614,102929.0
46487,Cosmos,4.330081,615.0
840,"Godfather, The (1972)",4.317030,66440.0
61090,Parasite (2019),4.312254,11670.0
50126,Blue Planet II (2017),4.300086,1163.0
59131,Twin Peaks (1989),4.298684,1140.0
68471,Twelve Angry Men (1954),4.286192,449.0



TOP WEIGHTED SCORE


,title,weighted_score,rating_mean,rating_count
314,"Shawshank Redemption, The (1994)",4.401238,4.404614,102929.0
40985,Planet Earth (2006),4.332311,4.444369,2948.0
840,"Godfather, The (1972)",4.312134,4.317030,66440.0
46103,Band of Brothers (2001),4.310914,4.426539,2811.0
61090,Parasite (2019),4.284956,4.312254,11670.0
46252,Planet Earth II (2016),4.284079,4.446830,1956.0
49,"Usual Suspects, The (1995)",4.260458,4.265070,67750.0
1190,"Godfather: Part II, The (1974)",4.257239,4.264468,43111.0
1173,12 Angry Men (1957),4.251126,4.265311,21863.0
522,Schindler's List (1993),4.232852,4.236990,73849.0


El baseline permite establecer una referencia inicial no personalizada para CineMatch. Se comparan recomendaciones por popularidad, rating promedio y score ponderado. El modelo ponderado reduce el sesgo hacia películas con pocas calificaciones y servirá como punto de comparación frente a los modelos personalizados posteriores.